In [8]:
import torch
import torch.nn as nn

In [9]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        self.W = nn.Parameter(torch.randn(self.in_features, self.out_features))
        self.b = nn.Parameter(torch.randn(self.out_features))
    
    def forward(self, x):
        # X shape: B, in_f
        x = x @ self.W + self.b
        return x

In [10]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

        self.W = nn.Parameter(torch.randn(self.num_embeddings, self.embedding_dim))

    def forward(self, x):
        return self.W[x]

In [11]:
config = {
    "context_length": 5,
    "vocab_size": 1000,
    "hidden_size": 50,
    "n_emb": 32
}

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.context_length = config["context_length"]
        self.vocab_size = config["vocab_size"]
        self.hidden_size = config["hidden_size"]
        self.n_emb = config["n_emb"]
        
        self.E = Embedding(self.vocab_size, self.n_emb)

        self.flat = nn.Flatten()
        self.fc = Linear(self.context_length * self.n_emb, self.hidden_size)
        self.tanh = nn.Tanh()
        self.out = Linear(self.hidden_size, self.vocab_size)

    def forward(self, x):
        
        x = self.E(x)
        x = self.flat(x)
        x = self.fc(x)
        x = self.tanh(x)

        return self.out(x)

In [12]:
model = MLP(config)

In [13]:
sum(p.nelement() for p in model.parameters())

91050

In [14]:
model

MLP(
  (E): Embedding()
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc): Linear()
  (tanh): Tanh()
  (out): Linear()
)

In [15]:
x = torch.randint(0, config["vocab_size"], (20, config["context_length"]))

In [16]:
logits = model(x)

In [17]:
logits

tensor([[ -5.6052,  -3.9755,   4.0221,  ...,  -7.5121,  -3.7957,   5.4890],
        [  3.9068,   6.4311,   6.7679,  ...,   3.8974,   2.1143,  -1.5699],
        [  9.0200,   8.4665,   3.9162,  ..., -15.7337,  -0.4377,  12.0878],
        ...,
        [ -3.0412,   5.0707,   8.4955,  ...,  -6.1486,  -4.7665,   2.8127],
        [  0.4515,   8.5346,   4.2416,  ...,   1.2127,   6.2021,  -5.4537],
        [  1.9545,  -1.6259,   9.6584,  ...,  10.5390, -13.2620,   3.3913]],
       grad_fn=<AddBackward0>)